# JFDS Experiment — 05: Appendix / Supplementary Analyses

**Part of a 5-notebook series.** This notebook holds the exploratory analyses that are
interesting but secondary to the core confirmatory claim (JFDS beats Top-K / MMR / Fairness-only
on the joint fairness-diversity objective — see `04_significance_and_robustness` for that result).

Contents:
- **R1** — Pareto frontier of JFDS recommendations in (Fairness, Diversity) space
- **R2** — Scaling law: what functional form best describes D as a function of F
- **R3** — Regime analysis: does the F→D relationship change slope across the F range
- **R4** — Conservation invariant: is there a stable F^α·D^β combination
- **R5** — Local coupling coefficient ρ_FD vs. utility
- **R6** — Information-theoretic checks (entropy vs. ILD, Gini vs. novelty)
- **Adaptive-λ schedule-shape sensitivity** — does the conclusion depend on the particular
  front-loaded/back-loaded schedule chosen for the adaptive-λ variant

None of these results are required to support the paper's central hypothesis; they're included
for readers who want a deeper look at the structure of the (F, D, U) trade-off space.

## Setup — load artifacts from `02_main_experiments.ipynb`

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations
from IPython.display import display, Markdown

with open('jfds_artifacts.pkl', 'rb') as f:
    ARTIFACTS = pickle.load(f)
globals().update(ARTIFACTS)

plt.rcParams['figure.dpi'] = 110
print(f"Loaded {len(ARTIFACTS)} artifacts from jfds_artifacts.pkl")
print(f"N_USERS={N_USERS}  N_ITEMS={N_ITEMS}  K={K}  POOL_SIZE={POOL_SIZE}")
print("best_lambdas:", best_lambdas)


In [ ]:
# -- Re-declare the core re-ranking + metric functions --
# These are pure functions of (tier, TARGET_SHARE, GENRE_SIM, genre_vec, pop_norm),
# all of which were just loaded from jfds_artifacts.pkl -- no retraining needed.
# Kept byte-identical to their definitions in 02_main_experiments.ipynb so results
# are guaranteed reproducible across notebooks.

def greedy_rerank(cand_ids, cand_rel, score_fn, k=K, **params):
    rel_norm = (cand_rel - cand_rel.min()) / (cand_rel.max() - cand_rel.min() + 1e-12)
    rel_map  = dict(zip(cand_ids, rel_norm))
    remaining = list(cand_ids)
    selected, tier_counts = [], {t: 0 for t in TIERS}

    for step in range(min(k, len(cand_ids))):
        best_score, best_item = -np.inf, None
        for c in remaining:
            s = score_fn(c, rel_map[c], selected, tier_counts, step, **params)
            if s > best_score:
                best_score, best_item = s, c
        selected.append(best_item)
        tier_counts[tier[best_item]] += 1
        remaining.remove(best_item)
    return selected

def topk_rerank(cand_ids, cand_rel, k=K):
    order = np.argsort(-cand_rel)
    return list(np.array(cand_ids)[order[:k]])

def fair_boost(cand_idx, tier_counts, n_selected):
    t = tier[cand_idx]
    current_share = tier_counts[t] / max(1, n_selected)
    gap = max(0.0, TARGET_SHARE[t] - current_share)
    return (gap ** 2) / (TARGET_SHARE[t] ** 2)

def diversity_term_fast(cand_idx, selected_idx):
    """Uses pre-computed GENRE_SIM -- no per-call cosine computation."""
    if not selected_idx:
        return 1.0
    return float(1.0 - GENRE_SIM[cand_idx, selected_idx].mean())

def jfds_score(cand_idx, rel_value, selected, tier_counts, step, lam_f, lam_d):
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def jfds_rerank(cand_ids, cand_rel, lam_f, lam_d, k=K):
    return greedy_rerank(cand_ids, cand_rel, jfds_score, k=k, lam_f=lam_f, lam_d=lam_d)

def mmr_max_sim(cand_idx, selected_idx):
    if not selected_idx:
        return 0.0
    return float(GENRE_SIM[cand_idx, selected_idx].max())

def mmr_score(cand_idx, rel_value, selected, tier_counts, step, lam=MMR_LAMBDA):
    return lam * rel_value - (1 - lam) * mmr_max_sim(cand_idx, selected)

def mmr_rerank(cand_ids, cand_rel, lam=MMR_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, mmr_score, k=k, lam=lam)

def fairness_only_score(cand_idx, rel_value, selected, tier_counts, step, lam_f=FAIR_ONLY_LAMBDA):
    return (1 - lam_f) * rel_value + lam_f * fair_boost(cand_idx, tier_counts, step)

def fairness_only_rerank(cand_ids, cand_rel, lam_f=FAIR_ONLY_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, fairness_only_score, k=k, lam_f=lam_f)

def ild(rec_list):
    """Intra-list diversity: 1 - mean pairwise genre-cosine similarity."""
    if len(rec_list) < 2:
        return 0.0
    idx = np.array(rec_list)
    sub = GENRE_SIM[np.ix_(idx, idx)]
    n = len(idx)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    return float(1.0 - sub[mask].mean())

def precision_recall_ndcg(rec_list, relevant_set, grades, k=K):
    rec   = rec_list[:k]
    hits  = len(set(rec) & relevant_set)
    prec  = hits / k
    rec_r = hits / max(1, len(relevant_set))
    dcg   = sum(grades.get(item, 0) / np.log2(r + 2) for r, item in enumerate(rec))
    idcg  = sum(g / np.log2(r + 2) for r, g in enumerate(sorted(grades.values(), reverse=True)[:k]))
    return prec, rec_r, (dcg / idcg if idcg > 0 else 0.0)

def novelty_fairness(rec_list):
    """Mean (1 - normalised popularity): higher = more novel/niche items recommended."""
    return float(np.mean([1 - pop_norm[i] for i in rec_list]))

def shannon_entropy(rec_list):
    dist = genre_vec[rec_list].mean(axis=0)
    dist = dist / dist.sum()
    return float(-np.sum([p * np.log2(p) for p in dist if p > 0]))

def gini(values):
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    if v.sum() == 0:
        return 0.0
    cum = np.cumsum(v)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def calibration_kl(rec_list, user_train_items, alpha=0.01):
    if len(user_train_items) == 0:
        return 0.0
    p = genre_vec[list(user_train_items)].mean(axis=0)
    p = p / p.sum()
    q = genre_vec[rec_list].mean(axis=0)
    q = (1 - alpha) * q + alpha * p
    q = q / q.sum()
    return float(np.sum(p * np.log((p + 1e-12) / (q + 1e-12))))

def aggregate_diversity(rec_lists_dict):
    all_items = set()
    for rec in rec_lists_dict.values():
        all_items.update(rec)
    return len(all_items)

def exposure_vector(rec_lists_dict, n_items=N_ITEMS):
    exp = np.zeros(n_items)
    for rec in rec_lists_dict.values():
        for i in rec:
            exp[i] += 1
    return exp

print('Core re-ranking + metric functions re-declared (greedy_rerank, topk_rerank, jfds_*, '
      'mmr_*, fairness_only_*, ild, precision_recall_ndcg, novelty_fairness, shannon_entropy, '
      'gini, calibration_kl, aggregate_diversity, exposure_vector)')


In [ ]:
# -- Adaptive-lambda schedule (same definition as in 02's "v3 Additions" section) --
DEFAULT_SCHEDULE = dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=1.0)

def schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p=1.0):
    """t_k = step / (k_total - 1); lambda grows from *_start to *_end as t_k^p."""
    t = step / max(1, k_total - 1)
    lam_f = lf_start + (lf_end - lf_start) * (t ** p)
    lam_d = ld_start + (ld_end - ld_start) * (t ** p)
    return lam_f, lam_d

def adaptive_jfds_score(cand_idx, rel_value, selected, tier_counts, step, k_total=K,
                         lf_start=DEFAULT_SCHEDULE['lf_start'], lf_end=DEFAULT_SCHEDULE['lf_end'],
                         ld_start=DEFAULT_SCHEDULE['ld_start'], ld_end=DEFAULT_SCHEDULE['ld_end'],
                         p=DEFAULT_SCHEDULE['p']):
    lam_f, lam_d = schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p)
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def adaptive_jfds_rerank(cand_ids, cand_rel, k=K, **schedule_params):
    params = {**DEFAULT_SCHEDULE, **schedule_params}
    return greedy_rerank(cand_ids, cand_rel, adaptive_jfds_score, k=k, k_total=k, **params)

print('Adaptive-lambda JFDS equation re-declared - default schedule:', DEFAULT_SCHEDULE)


In [ ]:
RNG = np.random.default_rng(RANDOM_SEED)
print('RNG re-seeded for appendix analyses ✓')

---
## R1 — Pareto Frontier (SVD vs. BPR)

In [ ]:
def pareto_mask(vals):
    n = len(vals)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        if dominated[i]:
            continue
        diff = vals - vals[i]
        better_eq = np.all(diff >= -1e-12, axis=1)
        strictly  = np.any(diff >  1e-9,  axis=1)
        dom = better_eq & strictly
        dom[i] = False
        if dom.any():
            dominated[i] = True
    return ~dominated

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
pareto_stats = {}

for ax, model_name, color in zip(axes, ['SVD', 'BPR'], [C_SVD, C_BPR]):
    jfds_pts = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')]
    topk_pts = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'TopK')]
    jfds_s = jfds_pts.sample(min(ANALYSIS_SAMPLE_N, len(jfds_pts)), random_state=RANDOM_SEED)
    topk_s = topk_pts.sample(min(ANALYSIS_SAMPLE_N, len(topk_pts)), random_state=RANDOM_SEED)

    vals = jfds_s[['F', 'D', 'U']].values
    mask = pareto_mask(vals)

    ax.scatter(topk_s['F'], topk_s['D'], s=14, c='lightgrey', alpha=0.5, label='Top-K (reference)')
    ax.scatter(jfds_s.loc[~mask, 'F'], jfds_s.loc[~mask, 'D'], s=16, c=color, alpha=0.35, label='JFDS (dominated)')
    ax.scatter(jfds_s.loc[ mask, 'F'], jfds_s.loc[ mask, 'D'], s=34, c=color,
               edgecolor='black', linewidth=0.5, label='JFDS (Pareto-optimal)')
    ax.set_xlabel('Fairness (F)'); ax.set_ylabel('Diversity (D)')
    ax.set_title(f'{model_name}: F-D Pareto Frontier', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')

    r_pareto = np.corrcoef(jfds_s.loc[mask, 'F'], jfds_s.loc[mask, 'D'])[0, 1] if mask.sum() > 2 else np.nan
    topk_vals = topk_s[['F', 'D', 'U']].values
    topk_dominated = sum(
        1 for i in range(len(topk_s))
        if (np.all(vals - topk_vals[i] >= -1e-12, axis=1) & np.any(vals - topk_vals[i] > 1e-9, axis=1)).any()
    )
    pareto_stats[model_name] = {
        'n_pareto_optimal_jfds': int(mask.sum()),
        'n_jfds_sampled': len(jfds_s),
        'pareto_pearson_r': r_pareto,
        'pct_topk_dominated_by_jfds': 100 * topk_dominated / len(topk_s),
    }

plt.tight_layout()
plt.savefig('r1_pareto_frontier.png', dpi=110, bbox_inches='tight')
plt.show()
pd.DataFrame(pareto_stats).T.round(3)


---
## R2 — Scaling Law (AIC/BIC)

In [ ]:
jfds_all = metrics_long[metrics_long.method == 'JFDS']
pool = jfds_all.sample(min(ANALYSIS_SAMPLE_N * 2, len(jfds_all)), random_state=RANDOM_SEED)
F_vals = pool['F'].values
D_vals = pool['D'].values
eps = 1e-6

def aic_bic(y, y_pred, k_params):
    n   = len(y)
    rss = max(np.sum((y - y_pred) ** 2), 1e-12)
    return n * np.log(rss/n) + 2 * k_params, n * np.log(rss/n) + k_params * np.log(n)

models_fit = {}
c1 = np.polyfit(F_vals, D_vals, 1)
models_fit['Linear'] = {'params': c1, 'pred': np.polyval(c1, F_vals), 'k': 2,
                        'eq': f"D = {c1[1]:.4f} + {c1[0]:.4f}*F"}
c2 = np.polyfit(F_vals, D_vals, 2)
models_fit['Quadratic'] = {'params': c2, 'pred': np.polyval(c2, F_vals), 'k': 3,
                           'eq': f"D = {c2[2]:.4f} + {c2[1]:.4f}*F + {c2[0]:.4f}*F^2"}

for name, fn, p0 in [
    ('Power',      lambda x, a, b: a * np.power(x + eps, b),          [1, 0.5]),
    ('Log',        lambda x, a, b: a + b * np.log(x + eps),            [0, 0.1]),
    ('Saturation', lambda x, a, b: a * x / (b + x + eps),              [1, 0.1]),
]:
    try:
        popt, _ = curve_fit(fn, F_vals, D_vals, p0=p0, maxfev=10000)
        models_fit[name] = {'params': popt, 'pred': fn(F_vals, *popt), 'k': 2,
                            'eq': f"{name} fit (a={popt[0]:.4f}, b={popt[1]:.4f})"}
    except RuntimeError:
        pass

rows_r2 = []
for name, m in models_fit.items():
    aic, bic = aic_bic(D_vals, m['pred'], m['k'])
    rows_r2.append({'Model': name, 'Equation': m['eq'], 'AIC': aic, 'BIC': bic})
scaling_table = pd.DataFrame(rows_r2).sort_values('AIC').reset_index(drop=True)
display(scaling_table)

best_name = scaling_table.iloc[0]['Model']
print(f"Best model by AIC: {best_name}  ->  {scaling_table.iloc[0]['Equation']}")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(F_vals, D_vals, s=8, alpha=0.25, color='grey', label='data (JFDS lists)')
F_grid = np.linspace(F_vals.min(), F_vals.max(), 200)
for (name, m), col in zip(models_fit.items(), plt.cm.tab10(np.linspace(0, 1, len(models_fit)))):
    try:
        if name in ('Linear', 'Quadratic'):
            yg = np.polyval(m['params'], F_grid)
        else:
            from scipy.optimize import curve_fit as _cf
            fn_map = {
                'Power':      lambda x, a, b: a * np.power(x + eps, b),
                'Log':        lambda x, a, b: a + b * np.log(x + eps),
                'Saturation': lambda x, a, b: a * x / (b + x + eps),
            }
            yg = fn_map[name](F_grid, *m['params'])
        lw = 2.5 if name == best_name else 1.2
        ax.plot(F_grid, yg, color=col, linewidth=lw,
                label=f"{name}{' (best)' if name == best_name else ''}")
    except Exception:
        pass
ax.set_xlabel('Fairness (F)'); ax.set_ylabel('Diversity (D)')
ax.set_title('R2 — Scaling Law: D as a function of F', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('r2_scaling_law.png', dpi=110, bbox_inches='tight')
plt.show()


---
## R3 — Regime Analysis

In [ ]:
pool = pool.copy()
pool['F_quintile'] = pd.qcut(pool['F'], 5, labels=[f'Q{i+1}' for i in range(5)], duplicates='drop')

slope_rows = []
for q, g in pool.groupby('F_quintile', observed=True):
    if len(g) < 3 or g['F'].std() < 1e-9:
        continue
    c, cov = np.polyfit(g['F'], g['D'], 1, cov=True)
    slope_rows.append({'quintile': q, 'F_range': f"[{g['F'].min():.3f}, {g['F'].max():.3f}]",
                       'n': len(g), 'slope': c[0], 'slope_se': np.sqrt(cov[0, 0])})

regime_df = pd.DataFrame(slope_rows)
display(regime_df.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(regime_df['quintile'].astype(str), regime_df['slope'],
       yerr=regime_df['slope_se'] * 1.96,
       color=[C_JFDS if s > 0 else C_TOPK for s in regime_df['slope']], capsize=4)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Fairness (F) quintile')
ax.set_ylabel('Local OLS slope  dD/dF  (±95% CI)')
ax.set_title('R3 — Regime Analysis', fontsize=12)
plt.tight_layout()
plt.savefig('r3_regime_analysis.png', dpi=110, bbox_inches='tight')
plt.show()

n_pos = int((regime_df['slope'] > 0).sum())
n_neg = int((regime_df['slope'] < 0).sum())
print(f"Positive-slope regimes: {n_pos}/{len(regime_df)}   Negative: {n_neg}/{len(regime_df)}")


---
## R4 — Conservation Invariant

In [ ]:
F_arr = pool['F'].values + 1e-6
D_arr = pool['D'].values + 1e-6
alphas = np.round(np.arange(-2, 2.01, 0.2), 2)
betas  = np.round(np.arange(-2, 2.01, 0.2), 2)

def cv_grid(F_arr, D_arr, alphas, betas):
    grid = np.zeros((len(alphas), len(betas)))
    for ai, a in enumerate(alphas):
        Fp = F_arr ** a
        for bi, b in enumerate(betas):
            Q  = Fp * (D_arr ** b)
            m  = np.mean(Q)
            grid[ai, bi] = np.std(Q) / abs(m) if abs(m) > 1e-9 else np.inf
    return grid

grid = cv_grid(F_arr, D_arr, alphas, betas)
best_idx = np.unravel_index(np.argmin(grid), grid.shape)
best_alpha, best_beta = alphas[best_idx[0]], betas[best_idx[1]]
best_cv = grid[best_idx]
print(f"Best invariant: F^{best_alpha} * D^{best_beta}   (CV = {best_cv:.4f})")

B, n = 200, len(F_arr)
boot_alpha, boot_beta = [], []
for _ in range(B):
    idx = RNG.integers(0, n, size=n)
    g   = cv_grid(F_arr[idx], D_arr[idx], alphas, betas)
    bi  = np.unravel_index(np.argmin(g), g.shape)
    boot_alpha.append(alphas[bi[0]]); boot_beta.append(betas[bi[1]])

alpha_ci = np.percentile(boot_alpha, [2.5, 97.5])
beta_ci  = np.percentile(boot_beta,  [2.5, 97.5])
print(f"Bootstrap 95% CI:  alpha [{alpha_ci[0]:.2f}, {alpha_ci[1]:.2f}]   "
      f"beta [{beta_ci[0]:.2f}, {beta_ci[1]:.2f}]")

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(grid.T, origin='lower',
               extent=[alphas.min(), alphas.max(), betas.min(), betas.max()],
               aspect='auto', cmap='viridis_r')
ax.scatter([best_alpha], [best_beta], color='red', marker='*', s=200,
           label=f'best (CV={best_cv:.3f})')
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel(r'$\beta$')
ax.set_title(r'R4 — CV of $F^\alpha D^\beta$', fontsize=12)
plt.colorbar(im, ax=ax, label='CV (lower = more conserved)')
ax.legend()
plt.tight_layout()
plt.savefig('r4_conservation_invariant.png', dpi=110, bbox_inches='tight')
plt.show()


---
## R5 — Coupling Coefficient ρ_FD

In [ ]:
order    = np.argsort(pool['F'].values)
F_sorted = pool['F'].values[order]
D_sorted = pool['D'].values[order]
U_sorted = pool['U'].values[order]
n, half  = len(F_sorted), 25

rho_fd = np.full(n, np.nan)
for pos in range(n):
    lo, hi = max(0, pos - half), min(n, pos + half + 1)
    Fw, Dw = F_sorted[lo:hi], D_sorted[lo:hi]
    if len(Fw) > 5 and np.var(Fw) > 1e-9:
        rho_fd[pos] = np.cov(Fw, Dw)[0, 1] / np.var(Fw)

valid = ~np.isnan(rho_fd)
r_coupling = np.corrcoef(rho_fd[valid], U_sorted[valid])[0, 1]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(U_sorted[valid], rho_fd[valid], s=10, alpha=0.35, color=C_JFDS)
c  = np.polyfit(U_sorted[valid], rho_fd[valid], 1)
xg = np.linspace(U_sorted[valid].min(), U_sorted[valid].max(), 100)
ax.plot(xg, np.polyval(c, xg), color='black', lw=2,
        label=f'trend (Pearson r={r_coupling:.3f})')
ax.axhline(0, color='grey', ls='--', lw=1)
ax.set_xlabel('Utility (U = NDCG@10)')
ax.set_ylabel(r'Local coupling $\rho_{{FD}}$ (50-NN slope)')
ax.set_title('R5 — Coupling Coefficient vs. Utility', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('r5_coupling_coefficient.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"Pearson r(rho_FD, Utility) = {r_coupling:.4f}")


---
## R6 — Information Theory

In [ ]:
H_vals        = pool['H'].values
one_minus_gini = 1 - pool['gini_exp'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

c1    = np.polyfit(H_vals, pool['D'].values, 1)
pred1 = np.polyval(c1, H_vals)
r2_1  = 1 - np.sum((pool['D'].values - pred1)**2) / np.sum((pool['D'].values - pool['D'].values.mean())**2)
axes[0].scatter(H_vals, pool['D'], s=10, alpha=0.3, color=C_SVD)
xg = np.linspace(H_vals.min(), H_vals.max(), 100)
axes[0].plot(xg, np.polyval(c1, xg), color='black', lw=2, label=f'R²={r2_1:.3f}')
axes[0].set_xlabel('Shannon entropy H'); axes[0].set_ylabel('Diversity (ILD)')
axes[0].set_title('Is D proportional to entropy?', fontsize=11)
axes[0].legend()

c2    = np.polyfit(one_minus_gini, pool['F'].values, 1)
pred2 = np.polyval(c2, one_minus_gini)
r2_2  = 1 - np.sum((pool['F'].values - pred2)**2) / np.sum((pool['F'].values - pool['F'].values.mean())**2)
axes[1].scatter(one_minus_gini, pool['F'], s=10, alpha=0.3, color=C_BPR)
xg2 = np.linspace(one_minus_gini.min(), one_minus_gini.max(), 100)
axes[1].plot(xg2, np.polyval(c2, xg2), color='black', lw=2, label=f'R²={r2_2:.3f}')
axes[1].set_xlabel('1 − Gini(popularity)'); axes[1].set_ylabel('Fairness (novelty)')
axes[1].set_title('Is F proportional to 1 − Gini?', fontsize=11)
axes[1].legend()

plt.suptitle('R6 — Information-Theoretic Checks', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('r6_information_theory.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"D ~ H:          R² = {r2_1:.4f}")
print(f"F ~ 1-Gini:     R² = {r2_2:.4f}")


---
## Adaptive-λ Schedule-Shape Sensitivity

Moved here from the main pipeline per the priority-3 restructuring: this check asks whether the
Adaptive-λ JFDS conclusions (see `04_significance_and_robustness`) depend on the particular
front-loaded / back-loaded / wide-range schedule chosen, rather than the linear default.

### E. Sensitivity Check — Does the λ-Schedule Shape Matter?

Repeat the Adaptive-λ comparison for a few alternative schedules (different `p`
exponent, different start/end weights) on a fixed sample of users per base model,
to see whether conclusions depend heavily on the particular schedule chosen.

In [ ]:
ALT_SCHEDULES = {
    'linear (default)':      dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=1.0),
    'front-loaded (p=0.5)':  dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=0.5),
    'back-loaded (p=2.0)':   dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=2.0),
    'wider range (0.0→0.6)': dict(lf_start=0.00, lf_end=0.60, ld_start=0.00, ld_end=0.40, p=1.0),
}

N_USERS_SENS = min(300, N_USERS)
rng_sens = np.random.default_rng(RANDOM_SEED)

sens_rows = []
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    sampled_u = rng_sens.choice(N_USERS, size=N_USERS_SENS, replace=False)
    for sched_name, sched_params in ALT_SCHEDULES.items():
        all_sel, niche_fracs, ilds, ndcgs = set(), [], [], []
        for u in sampled_u:
            cand_ids, cand_rel = pools[u]
            if len(cand_ids) == 0:
                continue
            rec = adaptive_jfds_rerank(cand_ids, cand_rel, **sched_params)
            all_sel.update(rec)
            niche_fracs.append(novelty_fairness(rec))
            ilds.append(ild(rec))
            relevant_set = test_relevant.get(u, set())
            grades = test_grades.get(u, {})
            _, _, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
            ndcgs.append(n_ndcg)
        fairness = float(np.mean(niche_fracs))
        agg_div  = len(all_sel) / N_ITEMS
        ild_mean = float(np.mean(ilds))
        jfds_composite = fairness * agg_div
        sens_rows.append({'base_model': model_name, 'Schedule': sched_name,
                          'Fairness': fairness, 'AggDiv': agg_div, 'ILD': ild_mean,
                          'JFDS_composite': jfds_composite, 'NDCG@K': float(np.mean(ndcgs))})
        print(f"[{model_name}] {sched_name:<26s} → Fairness={fairness:.4f}  AggDiv={agg_div:.4f}  "
              f"ILD={ild_mean:.4f}  JFDS={jfds_composite:.5f}  NDCG={np.mean(ndcgs):.4f}")

sens_df = pd.DataFrame(sens_rows)
display(sens_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
metrics_sens = ['Fairness', 'AggDiv', 'ILD', 'JFDS_composite']
for row, model_name in enumerate(['SVD', 'BPR']):
    sub = sens_df[sens_df.base_model == model_name]
    for col, metric in enumerate(metrics_sens):
        ax = axes[row][col]
        bars = ax.bar(sub['Schedule'], sub[metric], color=plt.cm.Set2(np.linspace(0, 1, len(sub))), edgecolor='black')
        for b, v in zip(bars, sub[metric]):
            ax.text(b.get_x() + b.get_width() / 2, v, f'{v:.3f}', ha='center', va='bottom', fontsize=7)
        ax.set_title(f'{model_name}: {metric}', fontsize=9)
        ax.tick_params(axis='x', rotation=30, labelsize=7)
plt.suptitle('Sensitivity of Adaptive-λ JFDS metrics to schedule shape', y=1.03, fontsize=12)
plt.tight_layout()
plt.savefig('sensitivity_schedule_shape.png', dpi=110, bbox_inches='tight')
plt.show()

### Appendix Summary

- **R1–R6** describe the *shape* of the (F, D, U) trade-off surface that JFDS operates in —
  Pareto-optimality, the best-fitting F→D scaling law, whether that relationship's slope is
  regime-dependent, whether an F^α·D^β quantity is approximately conserved, how strongly F and D
  are locally coupled as a function of utility, and whether D/F track classical information-theoretic
  proxies (entropy, Gini). These are descriptive/exploratory and do not bear directly on whether
  JFDS beats the baselines — that claim is tested in `04_significance_and_robustness`.
- **Schedule-shape sensitivity** (`sens_df`) shows whether the linear default schedule for
  Adaptive-λ JFDS is a reasonable default or whether conclusions are fragile to that choice.

**Note:** none of the numbers here should be read as ablation results — for the ablation study
(isolating JFDS's own fairness term, diversity term, and gap-exponent choice) see
`03_ablation_and_complexity.ipynb`.